In [1]:
!pip install -q pyannote.audio pyannote.metrics

import os, shutil, pathlib
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.

In [2]:
!ls /kaggle/input/          # confirm both slugs

datasets


In [3]:
import pathlib, shutil, os

ROOT = pathlib.Path("/kaggle/input")

# Find the directory that actually contains the scripts, wherever it landed.
CODE = next(p.parent for p in ROOT.rglob("stage3_diarize.py"))
# Find the directory that actually contains the WAVs.
AUDIO = next(p.parent for p in ROOT.rglob("*.wav"))
REF = next(p.parent for p in ROOT.rglob("clip_meta.csv"))
WORK = "/kaggle/working/data"

print("CODE  =", CODE)
print("AUDIO =", AUDIO, f"({len(list(AUDIO.glob('*.wav')))} wavs)")
print("REF   =", REF, f"({len(list(REF.rglob('*')))} files)")

CODE  = /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code
AUDIO = /kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav (99 wavs)
REF   = /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code/ref (205 files)


In [4]:
pathlib.Path(WORK).mkdir(parents=True, exist_ok=True)
shutil.copytree(REF, f"{WORK}/ref", dirs_exist_ok=True)
for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")
mf = next(ROOT.rglob("manifest.jsonl"), None)
if mf: shutil.copy(mf, WORK)

print("ref files:", len(list(pathlib.Path(f'{WORK}/ref').rglob('*'))))
print("scripts  :", [p.name for p in pathlib.Path('/kaggle/working').glob('*.py')])

ref files: 205
scripts  : ['stage3_diarize.py', 'build_notebooks.py', 'stage2_parse_refs.py', 'stage1_extract.py', 'stage3_score.py']


In [5]:
!pip show pyannote.audio | head -2

Name: pyannote-audio
Version: 4.0.7
ERROR: Pipe to stdout was broken
Exception ignored in: <_io.TextIOWrapper name='<stdout>' mode='w' encoding='utf-8'>
BrokenPipeError: [Errno 32] Broken pipe


In [6]:
!cd /kaggle/working && python stage3_diarize.py \
    --system pyannote31 --data data --wav-dir {AUDIO} --limit 5

[env ] system=pyannote31  model=pyannote/speaker-diarization-3.1
[env ] wav_dir=/kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav
[env ] device=cuda  gpu=Tesla T4
[run ] 5 of 5 clips to do (0 already done)

[env ] pyannote.audio 4.0.7
config.yaml: 100%|█████████████████████████████| 469/469 [00:00<00:00, 2.35MB/s]
pytorch_model.bin: 100%|███████████████████| 5.91M/5.91M [00:00<00:00, 10.8MB/s]
plda/xvec_transform.npz: 100%|████████████████| 134k/134k [00:00<00:00, 629kB/s]
plda/plda.npz: 100%|██████████████████████████| 134k/134k [00:00<00:00, 632kB/s]
pytorch_model.bin: 100%|███████████████████| 26.6M/26.6M [00:00<00:00, 63.5MB/s]
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_t

In [7]:
!python stage3_diarize.py --system pyannote31 --data data --wav-dir {AUDIO}

[env ] system=pyannote31  model=pyannote/speaker-diarization-3.1
[env ] wav_dir=/kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav
[env ] device=cuda  gpu=Tesla T4
[run ] 94 of 99 clips to do (5 already done)

[env ] pyannote.audio 4.0.7
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps

In [8]:
!python stage3_score.py --data data --systems pyannote31

[ok] scored pyannote31: 99 clips

STAGE 3 -- BASELINE DIARIZATION   (collar=0.0, skip_overlap=False, UEM=full clip)
system             DER    miss      FA    conf     JER  spk acc  spk MAE
------------------------------------------------------------------------------
pyannote31      27.34%  11.59%   5.89%   9.86%  38.14%    72.7%     0.33
------------------------------------------------------------------------------
DER/miss/FA/conf are duration-weighted. Macro (per-clip mean) for contrast:
  pyannote31     DER_macro  29.76%   JER_macro  38.09%   (weighted DER 27.34%)

------------------------------------------------------------------------------
DER by reference speaker count (duration-weighted within each bucket):
------------------------------------------------------------------------------
  pyannote31    2spk: 28.6%(n=25)  3spk: 26.7%(n=29)  4spk: 25.9%(n=28)  5spk: 33.6%(n=9)  6spk: 23.8%(n=4)  7spk: 25.0%(n=2)  8spk: 30.4%(n=2)

DER by overlap tercile:
--------------------------